In [9]:
# imports

from random import choice, shuffle
from lib.data_agents import ApiAgent
from tqdm import tqdm
import pandas as pd

In [10]:
# parameters

competition_limit = None

In [11]:
agent = ApiAgent()

competition_ids_endpoint = "model/Competition/id"
race_entity_endpoint_base = "model/race/entity/competition"
race_card_endpoint_base = "dataset/racecard"

In [12]:
competition_ids = agent.request_many(competition_ids_endpoint)
shuffle(competition_ids)
selection = competition_ids[:competition_limit]

races = []

print("Collecting races from competition...")
for id in tqdm(selection):
    race_endpoint = f"{race_entity_endpoint_base}/{id}"
    race_entities = agent.request_many(race_endpoint)
    for race in race_entities:
        races.append(race)


100%|██████████| 77187/77187 [04:17<00:00, 300.10it/s]


In [13]:
total_races = 0
added = 0
skipped = 0
race_cards = []

for race in tqdm(races):
    total_races += 1
    race_id = race["id"]
    race_card_endpoint = f"{race_card_endpoint_base}/{race_id}"
    result = agent.request_many(race_card_endpoint)
    if (len(result) == 0):
        skipped += 1
        continue
    race_cards.append(result)
    added += 1

print(f"Races: {total_races} | Added: {added} | skipped: {skipped}")

100%|██████████| 432552/432552 [40:34<00:00, 177.65it/s] 

Races: 432552 | Added: 63167 | skipped: 369385


In [14]:
participants = []
for item in race_cards:
    for p in item:
        participants.append(p)

print(f"Total participants: {len(participants)}")

Total participants: 638307


In [15]:
""""
if len(participants) > 0:
    item = choice(participants)
    print(item["date"])
    for key in item.keys(): print(f"{key}: {item[key]}")
"""
print()

In [16]:
dataframe = pd.DataFrame(participants)
dataframe.to_parquet("data/racecard_dataset.parquet", index=False)
dataframe.head();